# 05 — Yield Debug Analysis

**Prerequisite:** trained artifacts under `models/` (including `models/feature_engineering/corr_kept_cols.json`). Generate them from the repo root with `python src/train.py` after `data/processed/` exists (from the EDA / modeling notebooks).

This notebook demonstrates a **root-cause investigation workflow** for semiconductor yield failures.

Unlike notebooks 01–04 (EDA through SHAP explainability), this notebook focuses on
what an engineer does *after* the model is trained:

1. **Inspect predicted failures** — Which samples does the model flag, and with what confidence?
2. **Compare to healthy baseline** — How do failing wafers differ from normal production?
3. **Identify recurring sensor drivers** — Which sensors consistently appear as top contributors?
4. **Cluster failure patterns** — Do failures group into distinct failure modes?
5. **Generate diagnostic summaries** — Produce actionable per-sample and aggregate reports.

This supports the story that the project helps with **root-cause investigation**, not just prediction.

In [ ]:
import sys
import os
from pathlib import Path

# Repo root (works whether Jupyter cwd is `notebooks/` or project root)
ROOT = Path.cwd().resolve()
if (ROOT / "src" / "train.py").is_file():
    pass
elif (ROOT.parent / "src" / "train.py").is_file():
    ROOT = ROOT.parent
else:
    raise FileNotFoundError(
        "Could not find project root (expected a folder containing src/train.py). "
        "Open this notebook from the repo or `cd` into notebooks/ before running."
    )

sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

from src.diagnostics import DiagnosticsPipeline, generate_root_cause_summary

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

model_dir = ROOT / "models"
data_dir = ROOT / "data" / "processed"
marker = model_dir / "feature_engineering" / "corr_kept_cols.json"
if not marker.is_file():
    raise FileNotFoundError(
        f"Missing trained artifacts (e.g. {marker}).\n\n"
        "These files are created by the training script and are not stored in git.\n"
        "From the project root, with processed data in place, run:\n"
        "  python src/train.py\n\n"
        "Then re-run this cell."
    )

print("Loading pipeline artifacts...")
pipeline = DiagnosticsPipeline(model_dir=str(model_dir), data_dir=str(data_dir))
pipeline.load()
print(f"Loaded: {pipeline.n_samples} samples, {len(pipeline.fail_indices)} fails, "
      f"{len(pipeline.pass_indices)} passes")

## 1. Failure Triage — Model Predictions on All Samples

First, let's run the model on all known-fail samples and sort by predicted risk.
This is analogous to a yield engineer triaging a lot of flagged wafers.

In [ ]:
fail_diags = pipeline.batch_diagnose(pipeline.fail_indices, top_k=10)

triage_rows = []
for d in fail_diags:
    triage_rows.append({
        "Sample": d.sample_index,
        "Prediction": d.prediction,
        "Failure Prob": round(d.probability, 3),
        "Top Driver": d.top_shap_features[0],
        "Max |Z-Score|": round(float(d.deviations_z.abs().max()), 2),
    })

triage_df = pd.DataFrame(triage_rows).sort_values("Failure Prob", ascending=False)

print(f"Model correctly flagged {sum(1 for d in fail_diags if d.prediction == 'FAIL')} "
      f"of {len(fail_diags)} actual failures")
print(f"\nTop 20 highest-risk failures:")
triage_df.head(20)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(triage_df["Failure Prob"], bins=20, edgecolor="black", color="#d32f2f", alpha=0.7)
axes[0].set_xlabel("Predicted Failure Probability")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Failure Probabilities (Actual Fails)")
axes[0].axvline(x=0.5, color="black", linestyle="--", label="Decision threshold")
axes[0].legend()

axes[1].hist(triage_df["Max |Z-Score|"], bins=20, edgecolor="black", color="#ff9800", alpha=0.7)
axes[1].set_xlabel("Max |Z-Score| from Baseline")
axes[1].set_ylabel("Count")
axes[1].set_title("Maximum Sensor Deviation in Failed Wafers")
axes[1].axvline(x=3, color="black", linestyle="--", label="3σ threshold")
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Recurring Sensor Drivers Across Failures

Which sensors consistently show up as top model contributors in failing wafers?
If the same sensors keep appearing, they point to a systematic process issue
rather than random noise.

In [ ]:
pattern_summary = pipeline.get_failure_pattern_summary(top_k=10)
print("Top sensors by mean |SHAP| across all actual failures:")
pattern_summary

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

top_sensors = pattern_summary.head(10)
bars = ax.barh(top_sensors.index[::-1], top_sensors["Mean |SHAP|"].values[::-1],
               color="#1976d2", edgecolor="black", alpha=0.8)

ax.set_xlabel("Mean |SHAP| Value Across Failures")
ax.set_title("Most Consistent Failure Drivers (Aggregated Across All Failed Wafers)")

ax2 = ax.twiny()
ax2.barh(top_sensors.index[::-1], top_sensors["Frequency %"].values[::-1],
         color="#ff9800", alpha=0.3, edgecolor="none")
ax2.set_xlabel("Frequency in Top-10 (%)")

plt.tight_layout()
plt.show()

## 3. Baseline Deviation Analysis — Failed vs Healthy

For the top driver sensors, compare the distribution of values in failing wafers
against the healthy baseline. This tells us whether failures involve sensors
shifting high, low, or becoming more variable.

In [ ]:
top_driver_sensors = pattern_summary.head(6).index.tolist()

z_matrix = pd.DataFrame({
    d.sample_index: d.deviations_z[top_driver_sensors]
    for d in fail_diags
}).T

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=z_matrix, orient="h", palette="RdYlGn_r", ax=ax)
ax.axvline(x=0, color="black", linewidth=0.8)
ax.axvline(x=-2, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax.axvline(x=2, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax.set_xlabel("Z-Score (σ from healthy baseline)")
ax.set_title("Sensor Deviation Distribution in Failed Wafers")
plt.tight_layout()
plt.show()

In [ ]:
print("Median Z-scores for top sensors across failures:")
print("(Positive = sensor reading above baseline, Negative = below)")
print()
median_z = z_matrix.median().sort_values()
for sensor, z in median_z.items():
    direction = "HIGH" if z > 0 else "LOW"
    flag = " *** SYSTEMATIC" if abs(z) > 1 else ""
    print(f"  {sensor:>15s}: {z:+.2f}σ ({direction}){flag}")

## 4. Failure Mode Clustering

Do all failures look the same, or are there distinct failure modes?
We cluster the SHAP profiles of failed samples to see if failures
group into different patterns (e.g., one group driven by sensors 100–110,
another by sensors 400–420).

In [ ]:
shap_matrix = pd.DataFrame({
    d.sample_index: d.shap_values
    for d in fail_diags
}).T

shap_matrix = shap_matrix.fillna(0)

sil_scores = []
K_range = range(2, min(8, len(shap_matrix)))
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(shap_matrix)
    sil = silhouette_score(shap_matrix, labels)
    sil_scores.append(sil)

best_k = list(K_range)[np.argmax(sil_scores)]
print(f"Silhouette scores: {dict(zip(K_range, [round(s, 3) for s in sil_scores]))}")
print(f"Best k = {best_k}")

In [ ]:
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = km.fit_predict(shap_matrix)

pca = PCA(n_components=2, random_state=42)
shap_2d = pca.fit_transform(shap_matrix)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(shap_2d[:, 0], shap_2d[:, 1], c=cluster_labels,
                     cmap="Set1", s=60, alpha=0.7, edgecolors="black", linewidth=0.5)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title(f"Failure Mode Clusters (k={best_k}, SHAP-based)")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.show()

In [ ]:
print(f"Failure Mode Profiles (k={best_k} clusters)")
print("=" * 60)

for cluster_id in range(best_k):
    mask = cluster_labels == cluster_id
    cluster_shap = shap_matrix[mask]
    n_in_cluster = mask.sum()

    mean_abs_shap = cluster_shap.abs().mean().sort_values(ascending=False)
    top_3 = mean_abs_shap.head(3)

    print(f"\nCluster {cluster_id} ({n_in_cluster} samples, {n_in_cluster/len(shap_matrix)*100:.0f}%):")
    for sensor, val in top_3.items():
        median_z = z_matrix.loc[shap_matrix.index[mask], sensor].median() if sensor in z_matrix.columns else 0
        print(f"  Top driver: {sensor:>15s}  |  mean |SHAP| = {val:.4f}  |  median z = {median_z:+.2f}")

## 5. Deep-Dive: Single Sample Diagnostic Report

Demonstrate the full diagnostic workflow on the highest-risk sample.
This is what the Streamlit dashboard automates interactively.

In [ ]:
highest_risk = max(fail_diags, key=lambda d: d.probability)

print(f"Sample #{highest_risk.sample_index}")
print(f"Prediction: {highest_risk.prediction} ({highest_risk.probability:.1%} failure probability)")
print(f"Actual: FAIL")
print()

comparison = pipeline.get_baseline_comparison_df(highest_risk)
print("Sensor Comparison vs Healthy Baseline:")
print(comparison.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_feats = highest_risk.top_shap_features[:10]
shap_vals = [highest_risk.shap_values[f] for f in top_feats]
colors = ["#d32f2f" if v > 0 else "#1976d2" for v in shap_vals]

axes[0].barh(top_feats[::-1], shap_vals[::-1], color=colors[::-1], edgecolor="black")
axes[0].set_xlabel("SHAP Value")
axes[0].set_title(f"Top Sensor Drivers — Sample #{highest_risk.sample_index}")

z_vals = [highest_risk.deviations_z[f] for f in top_feats]
z_colors = ["#d32f2f" if abs(z) > 2 else "#ff9800" if abs(z) > 1 else "#4caf50" for z in z_vals]

axes[1].barh(top_feats[::-1], z_vals[::-1], color=z_colors[::-1], edgecolor="black")
axes[1].axvline(x=-2, color="gray", linestyle="--", alpha=0.5)
axes[1].axvline(x=2, color="gray", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Z-Score (σ from baseline)")
axes[1].set_title(f"Baseline Deviation — Sample #{highest_risk.sample_index}")

plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import Markdown, display

summary = generate_root_cause_summary(highest_risk, pipeline.baseline, top_k=5)
display(Markdown("### Preliminary Failure Pattern Summary"))
display(Markdown(summary))

## 6. Cross-Cluster Sensor Heatmap

Visualize which sensors drive failures in each cluster.
This helps identify whether different failure modes involve
different process steps or equipment.

In [ ]:
all_top_sensors = set()
for cluster_id in range(best_k):
    mask = cluster_labels == cluster_id
    cluster_top = shap_matrix[mask].abs().mean().sort_values(ascending=False).head(5).index
    all_top_sensors.update(cluster_top)

all_top_sensors = sorted(all_top_sensors)

heatmap_data = pd.DataFrame(index=[f"Cluster {i}" for i in range(best_k)],
                            columns=all_top_sensors, dtype=float)
for cluster_id in range(best_k):
    mask = cluster_labels == cluster_id
    for sensor in all_top_sensors:
        heatmap_data.loc[f"Cluster {cluster_id}", sensor] = float(
            shap_matrix.loc[shap_matrix.index[mask], sensor].abs().mean()
        )

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(heatmap_data.astype(float), annot=True, fmt=".3f", cmap="YlOrRd",
            ax=ax, linewidths=0.5)
ax.set_title("Mean |SHAP| by Cluster and Sensor — Failure Mode Signatures")
ax.set_ylabel("Failure Mode")
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated the yield debug workflow:

1. **Failure triage** — The model identifies high-risk wafers with probabilistic scores
2. **Recurring drivers** — A small set of sensors consistently drives failure predictions
3. **Baseline comparison** — Failing wafers show systematic shifts on specific sensors
4. **Failure clustering** — Distinct failure modes exist, each with different sensor signatures
5. **Diagnostic reports** — Per-sample summaries ground root-cause analysis in data

The interactive version of this workflow is available in the Streamlit dashboard:
```bash
streamlit run app/streamlit_app.py
```